In [9]:
# imports
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from transformers import pipeline
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
from edgar import Company, set_identity
import regex as re
import torch
import time
from io import StringIO
import hashlib

DATA_DIR = '../../../data'            # shared raw inputs and caches
OUT_DIR = '../../../outputs/v3/data'  # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)

### Gathering Data

In [10]:
# Universe: S&P 500. The list is saved on the first run and reused afterwards,
# so reruns use the same tickers the EODHD news was downloaded for.
UNIVERSE_PATH = f'{OUT_DIR}/sp500_tickers.csv'
if os.path.exists(UNIVERSE_PATH):
    table = pd.read_csv(UNIVERSE_PATH)
else:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    html = requests.get(url, headers=headers).text
    table = pd.read_html(html)[0]
    table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()
    table = table[['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']]
    table.to_csv(UNIVERSE_PATH, index=False)
table.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


In [ ]:
# Point-in-time membership: companies that were actually in the S&P 500 on that date
# Saves the results from Wikipedia (reruns won't change when Wikipedia is updated) for reproducibility

UNIVERSE_DATE = pd.Timestamp('2026-11-13')          # date sp500_tickers.csv was saved
CHANGES_PATH = f'{DATA_DIR}/sp500_changes.csv'      # in data/ so it is tracked by git
MEMBERSHIP_PATH = f'{OUT_DIR}/sp500_membership.csv'

if os.path.exists(CHANGES_PATH):
    changes = pd.read_csv(CHANGES_PATH, parse_dates=['Date'])
else:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
    url = 'https://en.wikipedia.org/wiki/Historical_components_of_the_S%26P_500'
    changes = pd.read_html(StringIO(requests.get(url, headers=headers).text))[0]
    changes.columns = ['Date', 'Added', 'AddedName', 'Removed', 'RemovedName', 'Reason', 'Refs']
    changes['Date'] = pd.to_datetime(changes['Date'], format='mixed')
    changes.to_csv(CHANGES_PATH, index=False)       # downloaded 2026-11-21

changes = changes[(changes['Date'] >= '2014-01-01') & (changes['Date'] <= UNIVERSE_DATE)].copy()
for col in ['Added', 'Removed']:
    changes[col] = changes[col].str.replace('.', '-', regex=False)

snapshots, members = [], set(table['Symbol'])
end = UNIVERSE_DATE + pd.Timedelta(days=1)
for d in sorted(changes['Date'].unique(), reverse=True):
    snapshots.append((pd.Timestamp(d), end, set(members)))     # members from d until end
    g = changes[changes['Date'] == d]
    members = (members - set(g['Added'].dropna())) | set(g['Removed'].dropna())   # undo the change
    end = pd.Timestamp(d)
snapshots.append((pd.Timestamp('2014-01-01'), end, set(members)))

dates = pd.bdate_range('2014-01-01', UNIVERSE_DATE)
all_ever = sorted(set().union(*(ms for _, _, ms in snapshots)))
in_index = pd.DataFrame(False, index=dates, columns=all_ever)
for s, e, ms in snapshots:
    in_index.loc[(dates >= s) & (dates < e), list(ms)] = True
in_index.to_csv(MEMBERSHIP_PATH)

print(in_index.sum(axis=1).describe())                        # should stay ~503-509 every day
print(f'{len(all_ever)} tickers ever in the index; '
      f'{len(set(all_ever) - set(table["Symbol"]))} removed since 2014')


count    2315.000000
mean      505.393089
std         1.603568
min       503.000000
25%       504.000000
50%       505.000000
75%       507.000000
max       509.000000
dtype: float64
672 tickers ever in the index; 169 removed since 2018


#### Tickers
The universe is the S&P 500 (current constituents, saved to `sp500_tickers.csv`).

The original 19 AI-sector tickers are kept in `ai_tickers` for subsample comparisons:<br>
**Semiconductors**: NVDA (NVIDIA), AMD (Advanced Micro Devices), INTC (Intel), ASML (ASML Holding), TSM (Taiwan Semiconductor Manufacturing), AVGO (Broadcom),
    MRVL (Marvell Technology), ARM (Arm Holdings), SMCI (Super Micro Computer)<br>
**Hyperscalers**: MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META (Facebook) <br>
**AI Pure Plays**: PLTR (Palantir), AI (C3.ai), SOUN (Soundhound AI) <br>
**Infrastructure**: DELL (Dell Technologies), ANET (Arista Networks), HPE (Hewlett Packard Enterprise)

In [12]:
# list of all the tickers with a dictionary that sorts them by type of stock
tickers = pd.read_csv(f'{DATA_DIR}/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
ai_tickers_flat = [t for group in ai_tickers.values() for t in group]
all_tickers = table['Symbol'].tolist()
print(f'{len(all_tickers)} tickers')

503 tickers


In [ ]:
# DO NOT RERUN: prices_raw_2014.csv was downloaded on 2026-11-27.
# # yfinance revises history, so redownloading would change the inputs. Uncomment only to rebuild everything.
# # Data from 2014 to today: 2015-2025 is the development sample, 2026 the holdout
# prices = yf.download(
#     tickers=all_tickers + ['SOXX', 'QQQ'],
#     start='2014-01-01',
#     end=(datetime.today() + timedelta(days=1)).strftime('%Y-%m-%d'),
#     auto_adjust=True,
#     progress=False
# )
# prices.to_csv(f'{OUT_DIR}/prices_raw_2014.csv')

In [ ]:
# Removed S&P 500 members since 2014 (point-in-time universe). Many delisted tickers will fail; that's expected.
changes_all = pd.read_csv(CHANGES_PATH, parse_dates=['Date'])
removed = set(changes_all.loc[(changes_all['Date'] >= '2014-01-01') & (changes_all['Date'] <= UNIVERSE_DATE), 'Removed']
              .dropna().str.replace('.', '-', regex=False)) - set(table['Symbol'])
print(len(removed), 'removed tickers')

REMOVED_PATH = f'{OUT_DIR}/prices_removed_2014.csv'
if os.path.exists(REMOVED_PATH):
    removed_prices = pd.read_csv(REMOVED_PATH, header=[0, 1], index_col=0, parse_dates=True)
else:
    removed_prices = yf.download(sorted(removed), start='2014-01-01', end='2026-11-28',
                                 auto_adjust=True, progress=False)
    removed_prices.to_csv(REMOVED_PATH)  
got = removed_prices['Close'].columns[removed_prices['Close'].notna().any()]
print(f'{len(got)} of {len(removed)} removed tickers have prices')

In [14]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# prices.head()

### Cleaning Data

In [15]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# close  = prices['Close']
# volume = prices['Volume']
# soxx   = close['SOXX']
# qqq    = close['QQQ']

In [16]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# # Remove tickers with more than 5% NaN values
# clean_prices = close.loc[:, close.isna().mean() <= 0.05]

# # Forward fill short gaps of 1-2 days
# clean_prices = clean_prices.ffill(limit=2)
# clean_prices = clean_prices.dropna()
# clean_prices.to_csv(f'{OUT_DIR}/clean_prices.csv')

In [17]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# # Convert prices to log
# log_ret = np.log(clean_prices / clean_prices.shift(1)).dropna()

### Feature Engineering

In [18]:
import sys
sys.path.insert(0, '../../../src')
sys.path.insert(0, '../../../scripts')
from features import make_features
from score_sentiment import assign_trading_date

In [19]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# all_feature_dfs = []
# for ticker in all_tickers:
#     if ticker not in close.columns:
#         continue
#     ticker_feats = make_features(
#         close  = close[ticker],
#         volume = volume[ticker],
#         soxx   = soxx,
#         qqq    = qqq
#     )
#     ticker_feats['Ticker'] = ticker      
#     all_feature_dfs.append(ticker_feats)

In [20]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# features_df = pd.concat(all_feature_dfs).reset_index()
# features_df = features_df.set_index(['Date','Ticker'])

In [21]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# # Forward 5-day cumulative return
# features_df['fwd_5d'] = (
#     features_df.groupby('Ticker')['ret_1d']
#     .transform(lambda x: x.rolling(5).sum().shift(-5))
# )

# features_df = features_df.dropna(subset=['fwd_5d'])

# # Cross-sectional: top half = 1, bottom half = 0 (guaranteed 50/50 per date)
# # Which stocks will outperform their peers? 
# features_df['Target'] = (
#     features_df.groupby('Date')['fwd_5d']
#     .transform(lambda x: (x > x.median()).astype(int))
# )

In [22]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# features_df.head()

In [23]:
# SKIPPED: features_df.csv is loaded from disk below (built 2026-11-14).
# features_df.to_csv(f'{OUT_DIR}/features_df.csv')

In [24]:
# Load the price features built on 2026-11-14 (the cells that rebuild them are commented out)
features_df = pd.read_csv(f'{OUT_DIR}/features_df.csv', index_col=['Date', 'Ticker'], parse_dates=['Date'])
print(features_df.shape, features_df.index.get_level_values('Date').min().date(), features_df.index.get_level_values('Date').max().date())

(965653, 14) 2019-01-02 2026-11-06


### News Sentiment (EODHD)
News is scored outside this notebook, because scoring ~500 tickers takes hours. Run these from the repository root, **after** the cell above saves `features_df.csv`:

```
caffeinate -i python scripts/download_news.py > logs/download_news.log 2>&1      # already done
caffeinate -i python scripts/score_sentiment.py > logs/score_sentiment.log 2>&1
```

`score_sentiment.py` assigns each article to the first trading day whose 4pm ET close comes after it was published, and writes `outputs/v3/data/sentiment_df.csv`.

In [25]:
# merge sentiment df back to features df
sentiment_df = pd.read_csv(f'{OUT_DIR}/sentiment_df.csv', parse_dates=['Date'])
news_cols = ['news_sentiment', 'news_count', 'pct_positive', 'eod_sentiment']

df = features_df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.merge(sentiment_df, on=['Date', 'Ticker'], how='left')
df = df.set_index(['Date', 'Ticker'])

# Fill missing sentiment (days with no news)
df[news_cols] = df[news_cols].fillna(0)

### SEC 8K Filings (Earnings Releases)

Filings are downloaded and scored by `scripts/score_filings.py` (zero-shot topic classification with DeBERTa, plus FinBERT tone on AI sentences). Run it from the repository root after `sp500_tickers.csv` exists:

```
python3 scripts/score_filings.py --time-only NVDA MMM                       # estimate run time
caffeinate -i python3 scripts/score_filings.py > logs/score_filings.log 2>&1
```

It saves each filing's text and scores per ticker, so it can be stopped and restarted, and writes `outputs/v3/data/filing_signals_df.csv`.

In [26]:
# Filing-level signals from scripts/score_filings.py
transcript_signals_df = pd.read_csv(f'{OUT_DIR}/filing_signals_df.csv')
text_cols = [col for col in transcript_signals_df.columns if col not in ('Ticker', 'Date', 'Accepted')]

# Earnings 8-Ks are often filed after the 4pm ET close. Count each filing on the first trading
# day whose close comes after the SEC accepted it (same rule as score_sentiment.py uses for news).
trading_days = features_df.index.get_level_values('Date').unique().sort_values()
transcript_signals_df['Date'] = assign_trading_date(transcript_signals_df['Accepted'], trading_days)
transcript_signals_df = transcript_signals_df.dropna(subset=['Date'])      # filed after the panel ends
transcript_signals_df['FilingDay'] = transcript_signals_df['Date']          # kept for the test

transcript_signals_df = (transcript_signals_df.drop(columns='Accepted')
                         .set_index(['Date', 'Ticker']).sort_index())

transcript_signals_df.to_csv(f'{OUT_DIR}/transcript_signals_df.csv')
transcript_signals_df.head()

ai_demand_up_share  ai_demand_down_share  capex_up_share  \
Date       Ticker                                                             
2019-01-02 A                      0.0                   0.0             0.0   
           A                      0.0                   0.0             0.0   
           A                      0.0                   0.0             0.0   
           A                      0.0                   0.0             0.0   
           AAPL                   0.0                   0.0             0.0   

                   capex_down_share  competition_share  supply_limited_share  \
Date       Ticker                                                              
2019-01-02 A                    0.0                0.0                   0.0   
           A                    0.0                0.0                   0.0   
           A                    0.0                0.0                   0.0   
           A                    0.0                0.0                   0.0   
           AAPL                 0.0                0.0                   0.0   

                   ai_demand_net  capex_net  ai_tone  ai_sentence_ratio  \
Date       Ticker                                                         
2019-01-02 A                 0.0        0.0      0.0                0.0   
           A                 0.0        0.0      0.0                0.0   
           A                 0.0        0.0      0.0                0.0   
           A                 0.0        0.0      0.0                0.0   
           AAPL              0.0        0.0      0.0                0.0   

                   FilingDay  
Date       Ticker             
2019-01-02 A      2019-01-02  
           A      2019-01-02  
           A      2019-01-02  
           A      2019-01-02  
           AAPL   2019-01-02

In [27]:
# merge back to original dataframe
df = df.reset_index()

df = pd.merge_asof(
    df.sort_values('Date'),
    transcript_signals_df.reset_index().rename(columns={'date': 'Date'}).sort_values('Date'),
    on='Date',
    by='Ticker',
    direction='backward',
    tolerance=pd.Timedelta(days=120) # Old filing scores become 0 after 1 quarter
)

with_domain_keywords = df.set_index(['Date', 'Ticker'])
with_domain_keywords[text_cols] = with_domain_keywords[text_cols].fillna(0)

### Final Cleaning

In [28]:
edgar_cols = text_cols

In [29]:
with_domain_keywords[edgar_cols] = with_domain_keywords[edgar_cols].fillna(0)

In [30]:
# Split development (to 2025) from the 2026 holdout, and seal the holdout
LAST_DEV_DAY = pd.Timestamp('2025-12-23')   # last day whose fwd_5d (t+1..t+5) ends by 2025-12-31.
                                            # Dropped: Dec 24, 26, 29, 30, 31, whose fwd_5d uses 2026 prices
HOLDOUT_START = pd.Timestamp('2026-01-01')
SEALED_DIR = '../../../outputs/v3/sealed'
RESEAL = False      # True only for a rebuild that is logged in PREREGISTRATION.md

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def split_and_seal(df, dev_name, sealed_name):
    dates = df.index.get_level_values('Date')
    dev, holdout = df[dates <= LAST_DEV_DAY], df[dates >= HOLDOUT_START]
    sealed_path = f'{SEALED_DIR}/{sealed_name}'
    if len(holdout):                                   # empty on reruns: the inputs are already development-only
        if os.path.exists(sealed_path) and not RESEAL:
            raise FileExistsError(f'{sealed_path} is already sealed. Set RESEAL = True only for a logged rebuild.')
        holdout.to_csv(sealed_path)
        with open(f'{SEALED_DIR}/SHA256SUMS.txt', 'a') as f:
            f.write(f'{sha256(sealed_path)}  {sealed_name}  sealed {pd.Timestamp.today().date()}  rows={len(holdout)}\n')
    dev.to_csv(f'{OUT_DIR}/{dev_name}')
    print(f'{dev_name}: {len(dev):,} rows, to {dev.index.get_level_values("Date").max().date()} | '
          f'{sealed_name}: {len(holdout):,} rows sealed')

split_and_seal(features_df, 'features_df.csv', 'features_2026.csv')
split_and_seal(with_domain_keywords, 'ai_sector_features_df.csv', 'ai_sector_features_2026.csv')

features_df.csv: 856,397 rows, to 2025-12-23 | features_2026.csv: 106,766 rows sealed
ai_sector_features_df.csv: 856,397 rows, to 2025-12-23 | ai_sector_features_2026.csv: 106,766 rows sealed


In [35]:
with_domain_keywords.head()

ret_1d    ret_3d    ret_5d   ret_10d   ret_20d    vol_5d  \
Date       Ticker                                                               
2019-01-02 MMM     0.002152  0.001311  0.069029 -0.009698 -0.081618  0.019782   
           COHR   -0.016020 -0.005913  0.079784 -0.027702 -0.146446  0.031812   
           CL     -0.004537 -0.007372  0.020847 -0.058328 -0.067223  0.011292   
           UAL     0.005374  0.013728  0.063953 -0.023547 -0.129473  0.026668   
           CMCSA   0.009398  0.002645  0.045091 -0.045536 -0.114044  0.014587   

                    vol_20d     rsi_14  rel_soxx   rel_qqq  ...  \
Date       Ticker                                           ...   
2019-01-02 MMM     0.019320  43.774078 -0.022212  0.001318  ...   
           COHR    0.028103  42.649455 -0.087040 -0.063510  ...   
           CL      0.014477  25.178504 -0.007817  0.015713  ...   
           UAL     0.025303  42.053255 -0.070067 -0.046537  ...   
           CMCSA   0.014900  31.970932 -0.054638 -0.031108  ...   

                   ai_demand_down_share  capex_up_share  capex_down_share  \
Date       Ticker                                                           
2019-01-02 MMM                      0.0        0.000000               0.0   
           COHR                     0.0        0.000000               0.0   
           CL                       0.0        0.000000               0.0   
           UAL                      0.0        0.000000               0.0   
           CMCSA                    0.0        0.015267               0.0   

                   competition_share  supply_limited_share  ai_demand_net  \
Date       Ticker                                                           
2019-01-02 MMM                   0.0                   0.0            0.0   
           COHR                  0.0                   0.0            0.0   
           CL                    0.0                   0.0            0.0   
           UAL                   0.0                   0.0            0.0   
           CMCSA                 0.0                   0.0            0.0   

                   capex_net  ai_tone  ai_sentence_ratio  FilingDay  
Date       Ticker                                                    
2019-01-02 MMM      0.000000      0.0                0.0 2019-01-02  
           COHR     0.000000      0.0                0.0 2019-01-02  
           CL       0.000000      0.0                0.0 2019-01-02  
           UAL      0.000000      0.0                0.0 2019-01-02  
           CMCSA    0.015267      0.0                0.0 2019-01-02  

[5 rows x 29 columns]

In [36]:
missing = set(table['Symbol']) - set(transcript_signals_df.index.get_level_values('Ticker').unique())
print(missing)


{'VYLR', 'ECHO'}
